In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import healpy as hp
import healsparse as hsp
import os
from astropy.io import fits
import matplotlib.pyplot as plt

nside = 8192


## Mask (in case it's necessary)

In [ ]:
def radec2thetaphi(ra, dec):
    """
    Convert ra and dec (in degrees) to theta and phi (Healpix conventions, in radians)

    """
    theta = np.pi / 2. - np.deg2rad(dec)
    phi = np.deg2rad(ra)
    return theta, phi

def make_mask_radec_cuts(nside, ramin, ramax, decmin, decmax, eps=0., dtype=float, nest=True):
    """Generate a square mask limited by RA/DEC

    Parameters
    ----------
    nside : int
        `nside` parameter for healpix.
    ramin, ramax : float
        Should be in degrees in [0., 360.]
    decmin, decmax: [type]
        Should be in degrees in [-90., +90.]

    Returns
    -------
    array
        Returns the square mask.
    """
    # Make sure arguments are correct
    assert (0. <= ramin <= 360.) & (0. <= ramax <= 360.) & (ramin < ramax)
    assert (-90. <= decmin <= 90.) & (-90. <= decmax <= 90.) & (decmin < decmax)

    mask = np.zeros(hp.nside2npix(nside))
    th, ph = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)), nest=nest)
    ths, phs = radec2thetaphi([ramin, ramax], [decmin, decmax])
    thmin, thmax = min(ths), max(ths)
    phmin, phmax = min(phs), max(phs)
    mask[np.where((th < thmax-eps) & (th > thmin+eps) & (ph > phmin+eps) & (ph < phmax-eps))[0]] = dtype(1)
    return mask
    
mask_octant = make_mask_radec_cuts(nside, 0, 90, 0, 90, eps=1e-10, nest=True) # mask fully within the octant
ipix_octant = np.where(mask_octant)[0]

hp.mollview(mask_octant, nest=True)


## List of all available simulations

In [ ]:
sim_seed_list = ["HACC150_3", "HACC195_2"] # simulation seed (different IC)
Nshells_list = [26, 34, 51, 101] # number of shells
seed_list = [0, 1, 2, 3, 4] # seed for the observer (5 pseudo-independent realizations per simulation seed)
imap_list = [0, 1, 2, 3, 4] # tomographic bin

path = "/pscratch/sd/j/jmena/HOS_Sims"


## Particle count maps

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[2] # whichever
seed = seed_list[0] # whichever

ishell_list = range(Nshells)
ishell = ishell_list[0] # whichever

# let's load it
particle_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/shell_hacc_seed{seed}_nside{nside}_imap{ishell}.fits")
particle_map = hsp.HealSparseMap.read(particle_filename)

# let's plot it
particle_map_healpy = particle_map.generate_healpix_map(nside=512, reduction='sum', nest=True)
hp.cartview(np.log10(1+particle_map_healpy), lonra=[0, 30], latra=[0, 30], nest=True, cmap=plt.cm.gray, cbar=False, title='')


## Shell $\kappa$ maps

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[0] # whichever
seed = seed_list[0] # whichever

ishell_list = range(Nshells)
ishell = ishell_list[0] # whichever

# let's load it
kappa_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/kappa_zsnapshots/kappa_hacc_seed{seed}_nside{nside}_imap{ishell}.fits")
kappa_map = hsp.HealSparseMap.read(kappa_filename)


## Shell $\gamma_1$ and $\gamma_2$ maps

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[0] # whichever
seed = seed_list[0] # whichever

ishell_list = range(Nshells)
ishell = ishell_list[0] # whichever

# let's load it
gamma1_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_gamma1_imap{ishell}.fits")
gamma1_map = hsp.HealSparseMap.read(gamma1_filename)

# let's load it
gamma2_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_gamma2_imap{ishell}.fits")
gamma2_map = hsp.HealSparseMap.read(gamma2_filename)


## SRD $\kappa$ maps

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[0] # whichever
seed = seed_list[0] # whichever
imap = imap_list[0] # whichever

# let's load it
kappa_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_imap{imap}.fits")
kappa_map = hsp.HealSparseMap.read(kappa_filename)


## SRD $\gamma_1$ and $\gamma_2$ maps

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[0] # whichever
seed = seed_list[0] # whichever
imap = imap_list[0] # whichever

# let's load it
gamma1_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_gamma1_imap{imap}.fits")
gamma1_map = hsp.HealSparseMap.read(gamma1_filename)

# let's load it
gamma2_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_gamma2_imap{imap}.fits")
gamma2_map = hsp.HealSparseMap.read(gamma2_filename)


## Galaxy catalogs

In [ ]:
sim_seed = sim_seed_list[0] # whichever
Nshells = Nshells_list[0] # whichever
seed = seed_list[0] # whichever
imap = imap_list[0] # whichever

catalog_filename = os.path.join(path, f"{sim_seed}/shells_z{Nshells}_subsampledensitymax1.5_nside{nside}/SRDv1_nz/kappa_hacc_seed{seed}_nside{nside}_catpoisson_interppixels_b1_imap{imap}.fits")
catalog = fits.open(catalog_filename)
data = catalog[1].data
